In [1]:
import os
import json
import numpy as np
import re
from scipy.stats import t, norm

In [17]:
# Función que calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n > 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n > 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"
        if n == 1:
            valor_str = f"{mean}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [3]:
def parse_valor_ic(valor):
    """Convierte un string '123.4 ± 2.3' en una tupla (valor, error)."""
    match = re.match(r"([-\d.]+)\s*±\s*([\d.]+)", valor)
    if match:
        valor = float(match.group(1))
        error = float(match.group(2))
        return valor, error
    else:
        raise ValueError(f"No se pudo parsear el valor con IC: {valor}")

def parse_valor_base(valor):
    """Convierte un string como '123.4' en float."""
    try:
        return float(valor)
    except Exception:
        raise ValueError(f"No se pudo convertir el valor base: {valor}")

def diferencia_intervalo(base, valor_ic_con_error):
    """Calcula (base - (valor + error), base - (valor - error))."""
    val_ic, error = parse_valor_ic(valor_ic_con_error)
    diff_min = round(base - (val_ic + error), 2)
    diff_max = round(base - (val_ic - error), 2)
    return (diff_min, diff_max)

def restar_dicts_base_menos_ic(dict_base, dict_ic):
    """Resta dict_base - dict_ic en todos los niveles, dejando tuplas (min, max)."""
    resultado = {}
    for key in dict_ic:
        if isinstance(dict_ic[key], dict) and isinstance(dict_base.get(key), dict):
            resultado[key] = restar_dicts_base_menos_ic(dict_base[key], dict_ic[key])
        elif isinstance(dict_ic[key], str) and isinstance(dict_base.get(key), str):
            try:
                base = parse_valor_base(dict_base[key])
                resultado[key] = diferencia_intervalo(base, dict_ic[key])
            except ValueError:
                resultado[key] = None
        else:
            resultado[key] = None
    return resultado

In [4]:
def escape_latex(s):
    """
    Escapa los caracteres especiales de LaTeX en un string.
    """
    if not isinstance(s, str):
        s = str(s)
    replacements = {
        "\\": r"\\textbackslash{}",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "$": r"\$",
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "^": r"\^{}",
        "~": r"\~{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements.keys()))
    return pattern.sub(lambda m: replacements[m.group()], s)

def dict_to_latex_string(d, indent=0):
    """
    Recursivamente convierte un dict anidado en un string con formato tipo código,
    escapando los caracteres peligrosos para LaTeX.
    """
    lines = []
    indent_str = "  " * indent
    for key, value in d.items():
        key_str = escape_latex(repr(key))
        if isinstance(value, dict):
            lines.append(f"{indent_str}{key_str}: "+"{")
            lines.extend(dict_to_latex_string(value, indent + 1))
            lines.append(f"{indent_str}"+"},")
        else:
            if isinstance(value, str):
                val_str = f"'{escape_latex(value)}'"
            else:
                val_str = escape_latex(repr(value))
            lines.append(f"{indent_str}{key_str}: {val_str},")
    return lines

def generar_latex_dict_texto(diccionario):
    """
    Devuelve un string listo para incluir en LaTeX.
    """
    return "\n".join(dict_to_latex_string(diccionario))

In [5]:
def save_nested_dict_tabbed_to_txt(d, filename="txt_kpis/kpis_output.txt"):
    os.makedirs("txt_kpis", exist_ok=True)

    lines = []

    def recurse(d, level=0):
        for i, (key, value) in enumerate(d.items()):
            indent = '\t' * level
            if isinstance(value, dict):
                lines.append(f"{indent}{key}:")
                recurse(value, level + 1)
                if level == 0:  # Add a blank line between top-level sections
                    lines.append("")
            else:
                lines.append(f"{indent}{key}: {value}")

    recurse(d)

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✅ Dict saved to '{filename}' in tabbed format.")

In [6]:
resumen_proactivo = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
resumen_base = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloBase_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
diferencias = restar_dicts_base_menos_ic(resumen_base, resumen_proactivo)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
save_nested_dict_tabbed_to_txt(diferencias)

✅ Dict saved to 'txt_kpis/kpis_output.txt' in tabbed format.


# Esto sirve para generar todos los resumenes de la sensibilidad

In [18]:
import os
import re

def procesar_y_replicar_kpis(origen_base, destino_base, nivel_confianza=99):
    for root, dirs, files in os.walk(origen_base):
        if os.path.basename(root) == 'kpis':
            padre = os.path.basename(os.path.dirname(root))
            match = re.fullmatch(r'\+(\d+)', padre)
            if match:
                numero = match.group(1)
                relative_parent_path = os.path.relpath(os.path.dirname(os.path.dirname(root)), origen_base)
                destino_dir = os.path.join(destino_base, relative_parent_path)
                os.makedirs(destino_dir, exist_ok=True)

                ruta_salida = os.path.join(destino_dir, f"+{numero}.json")
                try:
                    calcular_kpis_promedio_con_ic_en_formato_json(
                        directorio=root,
                        nivel_confianza=nivel_confianza,
                        guardar_en=ruta_salida
                    )
                    print(f"[✓] Resumen generado en: {ruta_salida}")
                except Exception as e:
                    print(f"[✗] Error en {root}: {e}")

In [ ]:
procesar_y_replicar_kpis(
    origen_base="resultados sensibilidad",
    destino_base="resumen resultados sensibilidad",
    nivel_confianza=99
)

In [22]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd

def plot_cost_vs_delta_por_unidad_bonito_guardado(df, carpeta_salida="resumen resultados sensibilidad/Plots"):
    """
    Genera y guarda un gráfico estilizado de costo vs delta para cada combinación de hospital y unidad.
    
    Parámetros:
    - df: DataFrame con columnas ['hospital', 'unidad', 'delta', 'costo']
    - carpeta_salida: ruta base donde guardar los gráficos generados
    """
    sns.set(style="whitegrid", font_scale=1.2)
    
    df['delta'] = pd.to_numeric(df['delta'], errors='coerce')
    df['costo'] = pd.to_numeric(df['costo'], errors='coerce')

    os.makedirs(carpeta_salida, exist_ok=True)
    
    # Agrupar por hospital y unidad
    grupos = df.groupby(['hospital', 'unidad'])

    for (hospital, unidad), grupo in grupos:
        grupo_ordenado = grupo.sort_values('delta')

        plt.figure(figsize=(10, 6))
        sns.lineplot(data=grupo_ordenado, x='delta', y='costo', marker='o', linewidth=2.5)
        plt.title(f"Hospital {hospital} - Unidad {unidad}", fontsize=16, weight='bold')
        plt.xlabel("Delta (camas adicionales)", fontsize=14)
        plt.ylabel("Costo total", fontsize=14)
        plt.xticks(grupo_ordenado['delta'], rotation=45)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()

        nombre_archivo = f"{hospital}_{unidad.replace('/', '-')}.png"
        ruta_salida = os.path.join(carpeta_salida, nombre_archivo)
        plt.savefig(ruta_salida, dpi=300)
        plt.close()

        print(f"[✓] Plot guardado en: {ruta_salida}")

# Cargar el DataFrame
df = pd.read_csv("resumen resultados sensibilidad/resultados_sensibilidad_completados.csv")

# Revisar las primeras filas
display(df.head())

plot_cost_vs_delta_por_unidad_bonito_guardado(df)

,hospital,unidad,delta,costo,timestamp,termino
0,1,OR,1,4427.78,2025-06-21T20:11:56.992947,NaN
1,1,OR,2,4347.28,2025-06-21T20:17:38.880646,NaN
2,1,OR,3,4302.74,2025-06-21T20:23:30.538679,NaN
3,1,OR,4,4302.52,2025-06-21T20:29:04.102626,NaN
4,1,OR,5,4328.77,2025-06-21T20:34:43.931770,cambio marginal


[✓] Plot guardado en: resumen resultados sensibilidad/Plots/1_ICU.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/1_OR.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/1_SDU-WARD.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/2_ICU.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/2_OR.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/2_SDU-WARD.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/3_ICU.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/3_OR.png
[✓] Plot guardado en: resumen resultados sensibilidad/Plots/3_SDU-WARD.png


In [23]:
def plot_resumen_por_hospital(df, carpeta_salida="resumen resultados sensibilidad/Plots"):
    """
    Genera un solo gráfico por hospital con las 3 unidades superpuestas (costo vs delta).
    
    Parámetros:
    - df: DataFrame con columnas ['hospital', 'unidad', 'delta', 'costo']
    - carpeta_salida: ruta donde guardar los plots
    """
    import seaborn as sns
    import matplotlib.pyplot as plt
    import os

    sns.set(style="whitegrid", font_scale=1.2)
    os.makedirs(carpeta_salida, exist_ok=True)

    df['delta'] = pd.to_numeric(df['delta'], errors='coerce')
    df['costo'] = pd.to_numeric(df['costo'], errors='coerce')

    hospitales = df['hospital'].unique()
    
    for hospital in hospitales:
        df_hosp = df[df['hospital'] == hospital]
        
        plt.figure(figsize=(12, 7))
        
        for unidad, grupo in df_hosp.groupby('unidad'):
            grupo = grupo.sort_values('delta')
            sns.lineplot(data=grupo, x='delta', y='costo', marker='o', label=unidad)

        plt.title(f"Resumen Hospital {hospital} - Costo vs Delta por Unidad", fontsize=16, weight='bold')
        plt.xlabel("Delta (camas adicionales)", fontsize=14)
        plt.ylabel("Costo total", fontsize=14)
        plt.legend(title="Unidad", fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()

        nombre_archivo = f"{hospital}_resumen.png"
        ruta_salida = os.path.join(carpeta_salida, nombre_archivo)
        plt.savefig(ruta_salida, dpi=300)
        plt.close()

        print(f"[✓] Plot resumen hospital guardado en: {ruta_salida}")

plot_resumen_por_hospital(df)

[✓] Plot resumen hospital guardado en: resumen resultados sensibilidad/Plots/1_resumen.png
[✓] Plot resumen hospital guardado en: resumen resultados sensibilidad/Plots/2_resumen.png
[✓] Plot resumen hospital guardado en: resumen resultados sensibilidad/Plots/3_resumen.png
